# 02 - Visual embedding (all videos)

??c to?n b? keyframes t? b??c 01, d?ng OpenCLIP m?t l?n tr?n GPU v? l?u vector ri?ng cho t?ng video.


In [ ]:
%pip install -q open_clip_torch pandas pyarrow pillow


In [ ]:
from __future__ import annotations

import gc
import json
import shutil
import traceback
from pathlib import Path

import numpy as np
import open_clip
import pandas as pd
import torch
from IPython.display import display
from PIL import Image

PIPELINE_VERSION = "aicv3-visual"
BATCH_NOTEBOOK_VERSION = "2026-08-18-visual-all-videos-v1"
MODEL_NAME = "ViT-B-32"
MODEL_PRETRAINED = "laion2b_s34b_b79k"
BATCH_SIZE = 64
INPUT_ROOT = Path("/kaggle/input")
BATCH_ROOT = Path("/kaggle/working/02-visual-embedding-all-videos-output")
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
KEYFRAME_PATHS = sorted(INPUT_ROOT.rglob("keyframes.parquet"))
if not KEYFRAME_PATHS:
    raise FileNotFoundError("No keyframes.parquet found in /kaggle/input")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    raise RuntimeError("Enable GPU in Kaggle Notebook Settings")
print("Batch notebook version:", BATCH_NOTEBOOK_VERSION)
print("Keyframe manifests found:", len(KEYFRAME_PATHS))
print("Device:", torch.cuda.get_device_name(0))


In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=MODEL_PRETRAINED, device=DEVICE
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
print("OpenCLIP model ready:", MODEL_NAME, MODEL_PRETRAINED)


In [ ]:
success_count = 0
failure_count = 0
failures = []

for video_number, KEYFRAMES_PATH in enumerate(KEYFRAME_PATHS, start=1):
    try:
        keyframes_df = pd.read_parquet(KEYFRAMES_PATH).sort_values("ordinal" if "ordinal" in pd.read_parquet(KEYFRAMES_PATH, columns=[]).columns else "frame_idx").reset_index(drop=True)
    except Exception:
        keyframes_df = pd.read_parquet(KEYFRAMES_PATH).sort_values("frame_idx").reset_index(drop=True)
    if keyframes_df.empty:
        print(f"[{video_number}/{len(KEYFRAME_PATHS)}] Skip empty manifest: {KEYFRAMES_PATH}")
        continue
    VIDEO_ID = str(keyframes_df.iloc[0]["video_id"])
    OUTPUT_ROOT = BATCH_ROOT / VIDEO_ID
    SUCCESS_PATH = OUTPUT_ROOT / "_SUCCESS.json"
    ERROR_PATH = OUTPUT_ROOT / "_ERROR.txt"
    if SUCCESS_PATH.exists():
        print(f"[{video_number}/{len(KEYFRAME_PATHS)}] Skip completed {VIDEO_ID}")
        success_count += 1
        continue

    print(f"\n[{video_number}/{len(KEYFRAME_PATHS)}] Processing {VIDEO_ID}")
    print("Manifest:", KEYFRAMES_PATH, "keyframes:", len(keyframes_df))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    INPUT_MANIFEST_ROOT = KEYFRAMES_PATH.parent
    frame_roots = list(INPUT_MANIFEST_ROOT.rglob("frames"))
    FRAME_ROOT = frame_roots[0] if frame_roots else INPUT_MANIFEST_ROOT / "frames"
    vectors = []
    missing = []

    try:
        with torch.inference_mode():
            for start in range(0, len(keyframes_df), BATCH_SIZE):
                batch_rows = keyframes_df.iloc[start:start + BATCH_SIZE]
                images = []
                valid_indices = []
                for index, row in batch_rows.iterrows():
                    frame_path = INPUT_MANIFEST_ROOT / str(row.frame_path)
                    if not frame_path.exists():
                        frame_path = FRAME_ROOT.parent / str(row.frame_path)
                    if not frame_path.exists():
                        candidates = list(INPUT_MANIFEST_ROOT.rglob(Path(str(row.frame_path)).name))
                        frame_path = candidates[0] if candidates else frame_path
                    if not frame_path.exists():
                        missing.append({"keyframe_id": row.keyframe_id, "frame_path": str(frame_path)})
                        continue
                    try:
                        images.append(preprocess(Image.open(frame_path).convert("RGB")))
                        valid_indices.append(index)
                    except Exception as exc:
                        missing.append({"keyframe_id": row.keyframe_id, "frame_path": str(frame_path), "error": str(exc)})
                if images:
                    batch = torch.stack(images).to(DEVICE)
                    encoded = torch.nn.functional.normalize(model.encode_image(batch).float(), dim=-1)
                    vectors.extend(encoded.cpu().numpy().astype("float32"))
                if start == 0 or start + len(batch_rows) == len(keyframes_df) or (start // BATCH_SIZE) % 4 == 0:
                    print(f"Embedding: {min(start + len(batch_rows), len(keyframes_df))}/{len(keyframes_df)}")

        if missing:
            raise RuntimeError(f"Missing or unreadable frames: {len(missing)}; examples: {missing[:3]}")
        visual_vectors = np.asarray(vectors, dtype="float32")
        if len(visual_vectors) != len(keyframes_df):
            raise RuntimeError(f"Vector count {len(visual_vectors)} != keyframe count {len(keyframes_df)}")
        if not np.isfinite(visual_vectors).all():
            raise RuntimeError("Embedding contains NaN or Inf")
        norms = np.linalg.norm(visual_vectors, axis=1)
        if not np.allclose(norms, 1.0, atol=1e-4):
            raise RuntimeError("Embedding vectors are not L2 normalized")

        np.save(OUTPUT_ROOT / "visual.npy", visual_vectors)
        records_df = keyframes_df[["keyframe_id", "video_id", "shot_id", "frame_idx", "pts_time", "frame_path"]].copy()
        records_df["embedding_row"] = np.arange(len(records_df), dtype="int64")
        records_df["embedding_model"] = f"{MODEL_NAME}/{MODEL_PRETRAINED}"
        records_df["embedding_dim"] = visual_vectors.shape[1]
        records_df.to_parquet(OUTPUT_ROOT / "embedding-records.parquet", index=False)
        summary = {
            "stage": "visual-embedding",
            "pipeline_version": PIPELINE_VERSION,
            "video_id": VIDEO_ID,
            "keyframes": int(len(keyframes_df)),
            "embedding_shape": list(visual_vectors.shape),
            "model": f"{MODEL_NAME}/{MODEL_PRETRAINED}",
            "metric": "IP with L2-normalized vectors",
        }
        (OUTPUT_ROOT / "visual-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        SUCCESS_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        ERROR_PATH.unlink(missing_ok=True)
        archive = shutil.make_archive(str(BATCH_ROOT / f"02-visual-embedding-{VIDEO_ID}"), "zip", root_dir=OUTPUT_ROOT)
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print("Vector norm sample:", norms[:5].round(5).tolist())
        print("Download:", archive)
        success_count += 1
    except Exception as exc:
        failure_count += 1
        ERROR_PATH.write_text(traceback.format_exc(), encoding="utf-8")
        failures.append({"video_id": VIDEO_ID, "error": f"{type(exc).__name__}: {exc}"})
        print(f"ERROR {VIDEO_ID}: {type(exc).__name__}: {exc}")
    finally:
        vectors = []
        globals().pop("visual_vectors", None)
        globals().pop("records_df", None)
        globals().pop("keyframes_df", None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

batch_summary = {"manifests_found": len(KEYFRAME_PATHS), "completed": success_count, "failed": failure_count, "failures": failures}
(BATCH_ROOT / "batch-summary.json").write_text(json.dumps(batch_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(batch_summary, ensure_ascii=False, indent=2))
if failure_count:
    print("Some videos failed. Inspect _ERROR.txt; rerunning retries videos without _SUCCESS.json.")


## Output

M?i video ???c l?u d??i `/kaggle/working/02-visual-embedding-all-videos-output/<VIDEO_ID>/`.
